# Lumis 10a: Real phone photos from SCIN

**SCIN** (Skin Condition Image Network, Google Research and Stanford Medicine, 2024) is a free dataset of about
10,000 skin photos that **people took themselves** (mostly on phones) and donated with informed consent.
Dermatologists labelled each case, including skin type. That is much closer to what Lumis users will upload than
DermNet's clinical photos.

Licence: SCIN Data Use License. Free to use and share with attribution; **never try to identify the people in the
photos**. Cite: Ward et al., "Crowdsourcing Dermatology Images with Google Search Ads: Creating a Real-World Skin
Condition Dataset", arXiv:2402.18545 (2024); dataset at github.com/google-research-datasets/scin.

This notebook:
1. downloads the SCIN cases whose **main dermatologist label** is one of Lumis's five conditions,
2. saves one close-up photo per case to `Drive/Lumis/data/scin/<condition>/` (notebook 10 uses them as "real conditions that must pass"),
3. runs the final model on them **as an external check**: how Lumis does on real phone photos it has never seen,
   both as uploaded and cropped to a centre square (to test the square framing guide).

These photos are **not** used to train or tune the model; the numbers are reported as they come out.
CPU runtime is fine. About 10 to 15 minutes, mostly downloading.

In [ ]:
MIN_AGREEMENT = 0.5      # the condition must carry at least half of the dermatologists' combined weight
MAX_PER_CLASS = 300      # cap on downloads per condition

In [ ]:
!pip install -q google-cloud-storage onnxruntime simplejpeg
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'

In [ ]:
import os, io, ast, json, time
import numpy as np, pandas as pd
from google.cloud import storage

BUCKET = 'dx-scin-public-data'
try:
    bucket = storage.Client.create_anonymous_client().bucket(BUCKET)
    bucket.blob('dataset/scin_cases.csv').download_as_bytes(start=0, end=10)
except Exception as e:
    print('Anonymous access failed, signing in instead (a popup will ask you to allow access):', type(e).__name__)
    from google.colab import auth
    auth.authenticate_user()
    bucket = storage.Client(project='dx-scin-public').bucket(BUCKET)

def fetch(path):
    return bucket.blob(path).download_as_bytes()

cases = pd.read_csv(io.BytesIO(fetch('dataset/scin_cases.csv')), dtype={'case_id': str})
labels = pd.read_csv(io.BytesIO(fetch('dataset/scin_labels.csv')), dtype={'case_id': str})
scin = cases.merge(labels, on='case_id')
print(len(scin), 'SCIN cases')

## 1. Map SCIN's condition names to Lumis's five classes

Lumis's eczema class comes from DermNet's eczema and atopic dermatitis folders, so contact dermatitis is **not**
counted as eczema. Check the printed mapping before going on.

In [ ]:
def parse(x):
    if isinstance(x, str) and x.strip():
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return {}
    return {}

scin['weights'] = scin.weighted_skin_condition_label.map(parse)
all_names = pd.Series([n for w in scin.weights for n in w]).value_counts()

def to_lumis(name):
    n = name.lower()
    if 'scabies' in n:
        return 'scabies'
    if 'psoria' in n and 'parapsoria' not in n:
        return 'psoriasis'
    if 'tinea' in n or 'dermatophyt' in n or 'ringworm' in n:
        return 'tinea'
    if n.startswith('acne') and 'rosacea' not in n and 'keloid' not in n:
        return 'acne'
    if ('eczema' in n or 'atopic' in n) and 'contact' not in n:
        return 'eczema'
    return None

mapping = {n: to_lumis(n) for n in all_names.index if to_lumis(n)}
print('SCIN names mapped to Lumis classes:')
for n, c in sorted(mapping.items(), key=lambda kv: kv[1]):
    print(f'  {c:10s} <- {n}  ({all_names[n]} cases mention it)')
print('\nMost common SCIN names (top 40), for reference:')
print(all_names.head(40).to_string())

In [ ]:
def main_label(w):
    if not w:
        return None, 0.0
    name, weight = max(w.items(), key=lambda kv: kv[1])
    return to_lumis(name), weight

scin[['lumis_class', 'agreement']] = scin.weights.apply(lambda w: pd.Series(main_label(w)))
chosen = scin[scin.lumis_class.notna() & (scin.agreement >= MIN_AGREEMENT)].copy()

def pick_image(r):
    for i in (1, 2, 3):                      # prefer the close-up shot
        if r.get(f'image_{i}_shot_type') == 'CLOSE_UP' and isinstance(r.get(f'image_{i}_path'), str):
            return r[f'image_{i}_path'], 'CLOSE_UP'
    for i in (1, 2, 3):
        if isinstance(r.get(f'image_{i}_path'), str):
            return r[f'image_{i}_path'], r.get(f'image_{i}_shot_type')
    return None, None

chosen[['image_path', 'shot_type']] = chosen.apply(lambda r: pd.Series(pick_image(r)), axis=1)
chosen = chosen[chosen.image_path.notna()]
chosen = chosen.groupby('lumis_class').head(MAX_PER_CLASS)

def fitz(r):
    vals = [r.get(f'dermatologist_fitzpatrick_skin_type_label_{i}') for i in (1, 2, 3)]
    vals = [v for v in vals if isinstance(v, str) and v.startswith('FST')]
    if not vals:
        return None
    v = int(round(np.median([int(x[3]) for x in vals])))
    return 'I-II' if v <= 2 else ('III-IV' if v <= 4 else 'V-VI')

chosen['tone'] = chosen.apply(fitz, axis=1)
print(chosen.lumis_class.value_counts().to_string())
print('\nSkin type (dermatologist estimate):')
print(pd.crosstab(chosen.lumis_class, chosen.tone.fillna('unknown')))

## 2. Download to Drive

In [ ]:
OUT = f'{BASE}/data/scin'
saved = []
t0 = time.time()
for _, r in chosen.iterrows():
    folder = f'{OUT}/{r.lumis_class}'
    os.makedirs(folder, exist_ok=True)
    ext = os.path.splitext(r.image_path)[1] or '.png'
    dest = f'{folder}/scin_{r.case_id}{ext}'
    if not os.path.exists(dest):
        try:
            open(dest, 'wb').write(fetch(r.image_path))
        except Exception as e:
            print('skipped', r.case_id, type(e).__name__)
            continue
    saved.append({'case_id': r.case_id, 'lumis_class': r.lumis_class, 'agreement': round(r.agreement, 3),
                  'tone': r.tone, 'shot_type': r.shot_type, 'file': dest})
saved = pd.DataFrame(saved)
saved.to_csv(f'{OUT}/scin_selected.csv', index=False)
print(f'{len(saved)} photos in {OUT} ({time.time() - t0:.0f} s)')

## 3. External check: Lumis on real phone photos

Run once and report as it comes out. "As uploaded" is the whole photo; "centre square" crops the middle square
first, which is what the app's framing guide will do.

In [ ]:
%%writefile predictor.py
"""Lumis skin-condition predictor.

Runs the exported EfficientNet-B0 (ONNX) and turns its output into what the app shows:
a condition, a calibrated confidence and a band (high / moderate / refer).

Preprocessing reproduces training exactly (ml/03_train.ipynb), checked pixel for pixel in ml/09_export.ipynb:
  1. Decode to RGB. JPEG uses libjpeg-turbo with the fast integer DCT, which is what
     tf.io.decode_image uses by default. PNG and WebP are decoded by Pillow.
  2. Resize to 224 x 224 like tf.image.resize: bilinear, half-pixel centers, no antialiasing,
     aspect ratio NOT kept (training stretched every image to a square).
  3. Cast to uint8 (truncation), then to float32 in the 0-255 range. EfficientNet scales and
     normalizes inside the model, so nothing else is done here.
One deliberate difference from training: phone photos are turned upright using their EXIF
orientation tag, because phones often save the pixels sideways.

Photo check (notebook 10), before any result is shown:
  no_skin           too few skin-colored pixels (the notebook 06 rule: YCbCr Cr 130-190, Cb 70-130)
  nothing_detected  the photo is unlike any of the training photos (energy score above the threshold)
Thresholds live in models/photo_check.json. Without that file every photo passes.

Output steps, decided in notebooks 04, 04b and 08:
  adjusted logits = logits + logit_adjust     (prior adjustment, training mix, tau 0.75)
  probabilities   = softmax(adjusted logits / temperature)
  band            = high if confidence >= high_from, moderate if >= refer_below, else refer
  scabies (any class in cap_at_moderate) is never shown as high: at most moderate.
"""
from __future__ import annotations

import hashlib
import io
import json
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import numpy as np
import simplejpeg
from PIL import Image, UnidentifiedImageError

MAX_PIXELS = 50_000_000          # about 8,000 x 6,000; larger uploads are refused
SUPPORTED_FORMATS = {"JPEG", "PNG", "WEBP"}


class UnsupportedImageError(ValueError):
    """The upload is not an image Lumis can read."""


# ---------------------------------------------------------------- preprocessing

def _apply_exif_orientation(rgb: np.ndarray, orientation: int) -> np.ndarray:
    """Same result as PIL.ImageOps.exif_transpose, on a numpy array."""
    if orientation == 2:
        return rgb[:, ::-1]
    if orientation == 3:
        return rgb[::-1, ::-1]
    if orientation == 4:
        return rgb[::-1]
    if orientation == 5:
        return rgb.transpose(1, 0, 2)
    if orientation == 6:
        return np.rot90(rgb, -1)
    if orientation == 7:
        return rgb.transpose(1, 0, 2)[::-1, ::-1]
    if orientation == 8:
        return np.rot90(rgb, 1)
    return rgb


def decode_image(data: bytes, apply_exif: bool = True) -> np.ndarray:
    """Bytes of a JPEG, PNG or WebP file -> uint8 array of shape (height, width, 3)."""
    try:
        im = Image.open(io.BytesIO(data))
        fmt = im.format
        if fmt not in SUPPORTED_FORMATS:
            raise UnsupportedImageError(f"Unsupported image format: {fmt}. Use JPEG, PNG or WebP.")
        if im.width * im.height > MAX_PIXELS:
            raise UnsupportedImageError("Image is too large.")
        orientation = im.getexif().get(0x0112, 1) if apply_exif else 1
        rgb = None
        if fmt == "JPEG":
            try:
                rgb = simplejpeg.decode_jpeg(data, colorspace="RGB", fastdct=True, fastupsample=False)
            except ValueError:
                rgb = None       # unusual JPEGs (for example CMYK): fall back to Pillow below
        if rgb is None:
            rgb = np.asarray(im.convert("RGB"))
    except UnsupportedImageError:
        raise
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError) as exc:
        raise UnsupportedImageError("The file could not be read as an image.") from exc
    return _apply_exif_orientation(rgb, orientation)


def resize_like_tf(img: np.ndarray, size: int) -> np.ndarray:
    """tf.image.resize(img, (size, size)) with its defaults: bilinear, half-pixel centers,
    no antialiasing. Returns float32."""
    h, w = img.shape[:2]

    def coords(n_in: int, n_out: int):
        src = (np.arange(n_out, dtype=np.float32) + 0.5) * np.float32(n_in / n_out) - 0.5
        lo = np.floor(src)
        frac = (src - lo).astype(np.float32)
        lo_i = np.clip(lo, 0, n_in - 1).astype(np.int64)
        hi_i = np.clip(lo + 1, 0, n_in - 1).astype(np.int64)
        return lo_i, hi_i, frac

    y0, y1, fy = coords(h, size)
    x0, x1, fx = coords(w, size)
    img = img.astype(np.float32)
    fx3, fy3 = fx[None, :, None], fy[:, None, None]
    top = img[y0][:, x0] * (1 - fx3) + img[y0][:, x1] * fx3
    bottom = img[y1][:, x0] * (1 - fx3) + img[y1][:, x1] * fx3
    return top * (1 - fy3) + bottom * fy3


def to_model_input(rgb: np.ndarray, size: int = 224) -> np.ndarray:
    """Decoded RGB array -> float32 model input of shape (1, size, size, 3), values 0-255."""
    resized = resize_like_tf(rgb, size).astype(np.uint8)       # truncation, as in training
    return resized.astype(np.float32)[None]


def preprocess(data: bytes, size: int = 224, apply_exif: bool = True) -> np.ndarray:
    """Image file bytes -> float32 model input of shape (1, size, size, 3), values 0-255."""
    return to_model_input(decode_image(data, apply_exif=apply_exif), size)


# ---------------------------------------------------------------- photo check

def skin_fraction(rgb: np.ndarray) -> float:
    """Share of skin-colored pixels, the rule from notebook 06: on a 128 x 128 copy, YCbCr chroma
    Cr 130-190 and Cb 70-130. Deliberately wide so red rashes and darker skin count as skin."""
    small = Image.fromarray(np.ascontiguousarray(rgb)).resize((128, 128)).convert("YCbCr")
    ycc = np.asarray(small, dtype=np.int16)
    cb, cr = ycc[..., 1], ycc[..., 2]
    return float(((cr >= 130) & (cr <= 190) & (cb >= 70) & (cb <= 130)).mean())


def energy_score(logits: np.ndarray) -> float:
    """Energy of the raw logits, -logsumexp (Liu et al., 2020). Photos like the training photos get low
    (more negative) energy; photos unlike anything the model was trained on get higher energy."""
    z = np.asarray(logits, dtype=np.float64).reshape(-1)
    m = z.max()
    return float(-(m + np.log(np.exp(z - m).sum())))


@dataclass(frozen=True)
class PhotoCheckConfig:
    min_skin_fraction: float
    max_energy: float

    @classmethod
    def load(cls, path: str | Path) -> "PhotoCheckConfig":
        c = json.loads(Path(path).read_text())
        return cls(min_skin_fraction=float(c["min_skin_fraction"]), max_energy=float(c["max_energy"]))

    def status(self, skin: float, energy: float) -> str:
        if skin < self.min_skin_fraction:
            return "no_skin"
        if energy > self.max_energy:
            return "nothing_detected"
        return "ok"


# ---------------------------------------------------------------- model output -> result

@dataclass(frozen=True)
class ModelConfig:
    model_version: str
    classes: list[str]
    img_size: int
    temperature: float
    logit_adjust: list[float]
    high_from: float
    refer_below: float
    cap_at_moderate: list[str] = field(default_factory=list)
    onnx_sha256: str | None = None
    model_file: str = "model.onnx"

    @classmethod
    def load(cls, path: str | Path) -> "ModelConfig":
        c = json.loads(Path(path).read_text())
        return cls(model_version=c["model_version"], classes=list(c["classes"]), img_size=int(c["img_size"]),
                   temperature=float(c["temperature"]), logit_adjust=[float(v) for v in c["logit_adjust"]],
                   high_from=float(c["bands"]["high_from"]), refer_below=float(c["bands"]["refer_below"]),
                   cap_at_moderate=list(c.get("cap_at_moderate", [])), onnx_sha256=c.get("onnx_sha256"),
                   model_file=c.get("model_file", "model.onnx"))


@dataclass(frozen=True)
class Prediction:
    condition: str              # most likely class (also given for 'refer', but the app shows it as inconclusive)
    confidence: float           # calibrated probability of that class, 0-1
    band: str                   # 'high' | 'moderate' | 'refer'
    probabilities: dict[str, float]
    capped: bool                # True if the scabies rule lowered the band from high to moderate
    model_version: str
    status: str = "ok"          # 'ok' | 'no_skin' | 'nothing_detected' (photo check)
    skin_fraction: float | None = None
    energy: float | None = None

    def to_dict(self) -> dict:
        return asdict(self)


def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def postprocess(logits: np.ndarray, cfg: ModelConfig) -> Prediction:
    """Raw model logits for one image (shape (K,)) -> Prediction."""
    logits = np.asarray(logits, dtype=np.float64).reshape(-1)
    probs = softmax((logits + np.asarray(cfg.logit_adjust)) / cfg.temperature)
    i = int(probs.argmax())
    conf = float(probs[i])
    condition = cfg.classes[i]
    if conf >= cfg.high_from:
        band = "high"
    elif conf >= cfg.refer_below:
        band = "moderate"
    else:
        band = "refer"
    capped = band == "high" and condition in cfg.cap_at_moderate
    if capped:
        band = "moderate"
    return Prediction(condition=condition, confidence=round(conf, 4), band=band,
                      probabilities={c: round(float(p), 4) for c, p in zip(cfg.classes, probs)},
                      capped=capped, model_version=cfg.model_version)


# ---------------------------------------------------------------- the predictor

def sha256_of(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


class LumisPredictor:
    """Load once at startup, then call predict() per upload. Safe to share across requests."""

    def __init__(self, model_path: str | Path, config_path: str | Path, verify_checksum: bool = True,
                 photo_check_path: str | Path | None = None):
        import onnxruntime as ort

        self.config = ModelConfig.load(config_path)
        self.photo_check = PhotoCheckConfig.load(photo_check_path) if photo_check_path else None
        if verify_checksum and self.config.onnx_sha256:
            actual = sha256_of(model_path)
            if actual != self.config.onnx_sha256:
                raise RuntimeError(f"Model file does not match its config (sha256 {actual[:12]}..., "
                                   f"expected {self.config.onnx_sha256[:12]}...). Re-download both from Drive.")
        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name

    @classmethod
    def from_dir(cls, models_dir: str | Path, config_name: str = "lumis_model_config.json") -> "LumisPredictor":
        """Load the config in models_dir and the ONNX file it names."""
        models_dir = Path(models_dir)
        config_path = models_dir / config_name
        photo_check = models_dir / "photo_check.json"
        return cls(models_dir / ModelConfig.load(config_path).model_file, config_path,
                   photo_check_path=photo_check if photo_check.exists() else None)

    def logits(self, batch: np.ndarray) -> np.ndarray:
        return self.session.run(None, {self.input_name: batch.astype(np.float32)})[0]

    def predict(self, data: bytes) -> Prediction:
        rgb = decode_image(data)
        skin = skin_fraction(rgb)
        logits = self.logits(to_model_input(rgb, self.config.img_size))[0]
        energy = energy_score(logits)
        status = self.photo_check.status(skin, energy) if self.photo_check else "ok"
        return replace(postprocess(logits, self.config), status=status,
                       skin_fraction=round(skin, 4), energy=round(energy, 4))

In [ ]:
import importlib, predictor
importlib.reload(predictor)
from predictor import LumisPredictor, UnsupportedImageError
from PIL import Image, ImageOps

pred = LumisPredictor.from_dir(f'{BASE}/exports/model')

def centre_square(data):
    im = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert('RGB')
    s = min(im.size)
    l, t = (im.width - s) // 2, (im.height - s) // 2
    b = io.BytesIO()
    im.crop((l, t, l + s, t + s)).save(b, 'JPEG', quality=95)
    return b.getvalue()

rows = []
for _, r in saved.iterrows():
    data = open(r.file, 'rb').read()
    for mode, d in [('as uploaded', data), ('centre square', None)]:
        try:
            res = pred.predict(d if d is not None else centre_square(data))
        except UnsupportedImageError:
            continue
        rows.append({'case_id': r.case_id, 'true': r.lumis_class, 'tone': r.tone, 'mode': mode,
                     'pred': res.condition, 'confidence': res.confidence, 'band': res.band,
                     'status': res.status, 'skin': res.skin_fraction, 'energy': res.energy})
ext = pd.DataFrame(rows)
ext['correct'] = ext.pred == ext.true
ext.to_csv(f'{OUT}/scin_external_results.csv', index=False)

def summarise(d):
    answered = d.band != 'refer'
    return pd.Series({'n': len(d), 'accuracy': d.correct.mean(), 'referred': 1 - answered.mean(),
                      'accuracy_when_answering': d[answered].correct.mean() if answered.any() else np.nan,
                      'high_share': (d.band == 'high').mean(),
                      'high_accuracy': d[d.band == 'high'].correct.mean() if (d.band == 'high').any() else np.nan})

print('Overall (photo check not applied here):')
display(ext.groupby('mode').apply(summarise).round(3))
print('\nSensitivity per condition:')
display(ext.pivot_table(index='true', columns='mode', values='correct', aggfunc='mean').round(3))
print('\nBy skin type (dermatologist estimate):')
display(ext.dropna(subset=['tone']).groupby(['mode', 'tone']).apply(summarise)[['n', 'accuracy', 'referred']].round(3))
print('\nWhat each condition was predicted as (centre square):')
display(pd.crosstab(ext[ext['mode'] == 'centre square'].true, ext[ext['mode'] == 'centre square'].pred, normalize='index').round(2))

In [ ]:
summary = {mode: summarise(d).round(4).to_dict() for mode, d in ext.groupby('mode')}
json.dump({'source': 'SCIN (Ward et al. 2024)', 'min_agreement': MIN_AGREEMENT, 'n_cases': int(len(saved)),
           'per_class': saved.lumis_class.value_counts().to_dict(), 'results': summary,
           'created_at': time.strftime('%Y-%m-%d %H:%M')},
          open(f'{OUT}/scin_external_summary.json', 'w'), indent=1, default=float)
print(json.dumps(summary, indent=1, default=float))